
# Лабораторна робота №2. Розпізнавання літери «Ф»

У цій роботі аналізується жест української дактильної абетки, який відповідає літері «Ф». Для розпізнавання використано MediaPipe Hand Landmarker, який будує тривимірний скелет кисті руки та повертає набір координат для 21 точки руки.

Мета реалізації полягає не в загальному підрахунку пальців, а в перевірці конкретної конфігурації кисті, яка відповідає опису літери «Ф»: долоня звернена до співбесідника, чотири пальці випрямлені та зближені, великий палець також випрямлений і притиснутий до ребра вказівного пальця.



## Опис жесту та ознак, які підлягають перевірці

Для літери «Ф» важливими є геометричні співвідношення між ключовими точками руки. У нормальному випадку вказівний, середній, безіменний і мізинець мають бути випрямлені, а відстані між їхніми кінчиками — невеликі. Великий палець не просто відкритий, а має бути орієнтований вгору та наближений до вказівного пальця.

Через це класична логіка «відкритий або закритий палець» недостатня. У коді нижче жест визначається як сукупність кількох правил: вертикальне положення пальців, близькість кінчиків чотирьох пальців між собою, а також окрема перевірка положення великого пальця.



## Технічні зауваження

Для запуску потрібен файл моделі `hand_landmarker.task`. Якщо його немає поруч із ноутбуком, код сам завантажить офіційну модель з публічного сховища. У процесі роботи використовується режим `VIDEO`, тому для кожного кадру передається timestamp, що зростає монотонно.

In [ ]:

import os
import time
import urllib.request
import cv2
import mediapipe as mp

# Reduce noisy backend logs where possible.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["GLOG_minloglevel"] = "2"

# Model settings
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
MODEL_PATH = "hand_landmarker.task"

# Download the model if it is missing.
if not os.path.exists(MODEL_PATH):
    print("Downloading MediaPipe Hand Landmarker model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

# Camera settings
cap = cv2.VideoCapture(0)
wCam, hCam = 640, 480
cap.set(cv2.CAP_PROP_FRAME_WIDTH, wCam)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, hCam)

# MediaPipe Tasks API
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=RunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.8,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

# Landmark indices used by MediaPipe Hands.
TIP_IDS = [4, 8, 12, 16, 20]

# Manual connections for drawing the hand skeleton.
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (17, 18), (18, 19), (19, 20),
    (0, 17)
]



## Алгоритм розпізнавання

Спочатку з кадру будується набір landmark-точок руки. Далі для чотирьох пальців перевіряється, чи їхні кінчики розташовані вище за відповідні середні фаланги. Після цього перевіряється, чи ці чотири пальці достатньо близько один до одного. Для великого пальця окремо контролюється його підняте положення над іншими пальцями та близькість до вказівного пальця.

Якщо всі умови виконані, кадр вважається таким, що містить жест «Ф». Це правило добре узгоджується з текстовим описом жесту, але чутливе до освітлення, відстані до камери та часткових перекриттів пальців.


In [ ]:
def distance(lm_a, lm_b):
    # Return Euclidean distance between two normalized landmarks.
    return ((lm_a.x - lm_b.x) ** 2 + (lm_a.y - lm_b.y) ** 2) ** 0.5

def finger_is_extended(landmarks, tip_idx, pip_idx, wrist_idx=0, margin=0.02):
    # Check whether a finger is extended by comparing its tip distance to the wrist.
    tip = landmarks[tip_idx]
    pip = landmarks[pip_idx]
    wrist = landmarks[wrist_idx]
    return distance(tip, wrist) > distance(pip, wrist) + margin

def thumb_matches_letter_f(landmarks, distance_margin=0.03, close_margin=0.11, height_margin=0.02):
    # The thumb should be extended upward and pressed close to the index finger side.
    thumb_tip = landmarks[4]
    thumb_ip = landmarks[3]
    wrist = landmarks[0]

    index_tip = landmarks[8]
    index_pip = landmarks[6]
    middle_tip = landmarks[12]
    ring_tip = landmarks[16]
    pinky_tip = landmarks[20]

    thumb_extended = distance(thumb_tip, wrist) > distance(thumb_ip, wrist) + distance_margin

    thumb_above_all_fingers = (
        thumb_tip.y < min(index_tip.y, middle_tip.y, ring_tip.y, pinky_tip.y) - height_margin
    )

    thumb_close_to_index = (
        distance(thumb_tip, index_tip) < close_margin
        or distance(thumb_ip, index_pip) < close_margin
    )

    return thumb_extended and thumb_above_all_fingers and thumb_close_to_index

def fingers_are_close_together(landmarks, max_tip_gap=0.09, max_y_spread=0.07):
    # Check whether index, middle, ring, and pinky fingertips are compact.
    tips = [landmarks[i] for i in [8, 12, 16, 20]]

    adjacent_gaps = [
        distance(tips[0], tips[1]),
        distance(tips[1], tips[2]),
        distance(tips[2], tips[3]),
    ]
    y_values = [lm.y for lm in tips]

    return (
        max(adjacent_gaps) < max_tip_gap
        and (max(y_values) - min(y_values)) < max_y_spread
    )

def is_letter_f(landmarks):
    # Decide whether the detected hand matches the Ukrainian letter 'Ф'.
    index_open = finger_is_extended(landmarks, 8, 6)
    middle_open = finger_is_extended(landmarks, 12, 10)
    ring_open = finger_is_extended(landmarks, 16, 14)
    pinky_open = finger_is_extended(landmarks, 20, 18)

    four_fingers_open = index_open and middle_open and ring_open and pinky_open
    fingers_compact = fingers_are_close_together(landmarks)
    thumb_ok = thumb_matches_letter_f(landmarks)

    return four_fingers_open and fingers_compact and thumb_ok

def draw_hand_landmarks(image, hand_landmarks):
    # Draw landmarks and hand connections on the frame.
    if not hand_landmarks:
        return

    h, w, _ = image.shape

    for landmarks in hand_landmarks:
        for lm in landmarks:
            cx, cy = int(lm.x * w), int(lm.y * h)
            cv2.circle(image, (cx, cy), 5, (0, 255, 0), cv2.FILLED)

        for start_idx, end_idx in HAND_CONNECTIONS:
            start = landmarks[start_idx]
            end = landmarks[end_idx]
            x1, y1 = int(start.x * w), int(start.y * h)
            x2, y2 = int(end.x * w), int(end.y * h)
            cv2.line(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

def annotate_frame(image, detected_f):
    # Add status text to the output frame.
    text = "Letter: F" if detected_f else "Letter: -"
    color = (0, 255, 0) if detected_f else (0, 0, 255)
    cv2.putText(image, text, (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)


## Запуск відеопотоку

Нижче наведено основний цикл роботи. Для кожного кадру виконується дзеркальне відображення, передача до моделі, побудова скелета руки та перевірка на жест «Ф». Для виходу з програми використовується клавіша пробіл.

Якщо камера відкрита правильно, у вікні буде видно руку, нанесені точки скелета та текстовий індикатор розпізнаного стану.


In [ ]:

frame_idx = 0

with HandLandmarker.create_from_options(options) as landmarker:
    while True:
        success, image = cap.read()
        if not success:
            break

        image = cv2.flip(image, 1)
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_image
        )

        # VIDEO mode requires a monotonically increasing timestamp.
        timestamp_ms = frame_idx * 33
        frame_idx += 1

        result = landmarker.detect_for_video(mp_image, timestamp_ms)
        hand_landmarks = result.hand_landmarks

        detected_f = False
        if hand_landmarks:
            detected_f = is_letter_f(hand_landmarks[0])

        draw_hand_landmarks(image, hand_landmarks)
        annotate_frame(image, detected_f)

        cv2.imshow("Output", image)

        key = cv2.waitKey(1)
        if key == 32:
            break

cap.release()
cv2.destroyAllWindows()



## Інтерпретація результатів

Умови освітлення, контраст фону, відстань до камери та повнота видимості кисті мають прямий вплив на стабільність розпізнавання. Найкращий результат зазвичай отримується тоді, коли долоня повністю видима, пальці не перекривають один одного, а камера не має сильних відблисків або рухового змазування.

Для літери «Ф» особливо критичним є збереження форми чотирьох випрямлених і зближених пальців та коректне положення великого пальця. 